# 🌐 Tanglish DistilGPT2 — Fine-Tuning on Kaggle (v2)

**GPU runtime required.** Enable via *Settings → Accelerator → GPU T4 x2 or P100.*
**Internet must be ON.** Enable via *Settings → Internet → On* — this notebook pulls the dataset directly from HuggingFace, it does not use a local Kaggle Dataset input.

> **What changed from v1, and why:**
> - Loads `vishnu-n/Tanglish-Corpus-185k` (185,973 real Tanglish sentences from YouTube/Reddit) directly from HuggingFace, instead of a small local CSV.
> - Adds a real preprocessing pipeline: cleaning, length filtering, domain balancing (so YouTube/political-comment sources don't dominate), and exact + near-duplicate removal.
> - Tokenizes **per sentence with an EOS separator**, instead of concatenating all text and chopping it into fixed 128-token blocks. Block-packing without separators teaches the model that one sentence's end flows straight into an unrelated sentence's start — exactly wrong for a short-message autocomplete feature.
> - Evaluates every 500 steps instead of only once per epoch, since with ~180k rows a full epoch is a lot of steps to go without checking for overfitting.
> - Uses gradient accumulation to reach a larger effective batch size without exceeding Kaggle's GPU memory.

> **Workflow:** Run All. At the end, download `tanglish-distilgpt2.zip` from the Output panel.


In [ ]:
# ============================================================
#  ✏️  CONFIGURE THIS CELL BEFORE RUNNING
# ============================================================

# HuggingFace dataset to pull directly (requires Internet ON)
HF_DATASET_NAME = "vishnu-n/Tanglish-Corpus-185k"

MODEL_NAME = "distilgpt2"

# --- Preprocessing ---
MAX_SOURCE_SHARE   = 0.10   # cap any single `source` value at 10% of the cleaned corpus
MIN_CHARS          = 15
MAX_CHARS          = 250
MIN_TOKENS         = 4
MAX_TOKENS         = 40
NON_LATIN_MAX_RATIO = 0.10  # drop rows that are >10% non-Latin Unicode (leftover Tamil script)
NEAR_DUP_THRESHOLD  = 0.92  # difflib similarity ratio above which two rows count as near-duplicates
VAL_FRACTION        = 0.03  # 97/3 split — with 180k+ rows, 3% still gives a solid val set

# --- Tokenization ---
MAX_LENGTH = 64   # covers the corpus's ~14-token average sentence with generous headroom

# --- Training ---
NUM_EPOCHS               = 4
PER_DEVICE_TRAIN_BATCH   = 16
PER_DEVICE_EVAL_BATCH    = 16
GRAD_ACCUM_STEPS         = 4     # effective batch size = 16 * 4 = 64
LEARNING_RATE            = 3e-5  # lower than a small-dataset run: ~60x more data, more steps, less LR needed for stability
WARMUP_RATIO             = 0.06
WEIGHT_DECAY             = 0.01
EVAL_STEPS               = 500
SAVE_STEPS               = 500
SAVE_TOTAL_LIMIT         = 2

# Output paths (Kaggle working dir is persisted → downloadable)
OUTPUT_MODEL_DIR = "/kaggle/working/tanglish-distilgpt2"
OUTPUT_ZIP       = "/kaggle/working/tanglish-distilgpt2.zip"

import os
print(f"Dataset      : {HF_DATASET_NAME}")
print(f"Model output : {OUTPUT_MODEL_DIR}")


In [ ]:
# ── Install / upgrade dependencies ──────────────────────────
# Kaggle already has torch; we only upgrade the HF stack.
!pip install -q --upgrade transformers datasets accelerate


In [ ]:
# ── Imports ──────────────────────────────────────────────────
import os, re, math, random, zipfile, difflib
from collections import defaultdict

import torch
from datasets import load_dataset, Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed,
)

set_seed(42)
random.seed(42)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    print(f"VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")


## Load the dataset directly from HuggingFace

No local CSV — this pulls the full 185,973-row corpus straight from the Hub. Requires Internet to be ON in notebook settings.


In [ ]:
# ── Load raw dataset from HuggingFace ────────────────────────
raw = load_dataset(HF_DATASET_NAME)["train"]
print(raw)
print(raw[0])

df = raw.to_pandas()
print(f"\nLoaded rows: {len(df):,}")
print("\nSource value counts (top 15):")
print(df["source"].value_counts().head(15))


## Clean the text

Strip URLs and markdown/meta artifacts, collapse repeated punctuation, and drop rows that are mostly non-Latin Unicode (leftover native-script Tamil, which the tokenizer/model has almost no exposure to).


In [ ]:
# ── Cleaning ─────────────────────────────────────────────────
URL_RE       = re.compile(r"(https?://\S+|www\.\S+)")
MD_NOISE_RE  = re.compile(r"(\*\*|/s\b|\\\*+)")
REPEAT_PUNCT_RE = re.compile(r"([!?.,])\1{2,}")
WS_RE        = re.compile(r"\s+")

def non_latin_ratio(text: str) -> float:
    if not text:
        return 1.0
    non_latin = sum(1 for ch in text if ord(ch) > 0x2FF and not ch.isspace())
    return non_latin / max(len(text), 1)

def clean_text(text: str) -> str:
    text = URL_RE.sub("", text)
    text = MD_NOISE_RE.sub("", text)
    text = REPEAT_PUNCT_RE.sub(r"\1", text)
    text = WS_RE.sub(" ", text).strip()
    return text

df["text"] = df["text"].astype(str).map(clean_text)
df = df[df["text"].str.len() > 0]
df = df[df["text"].map(non_latin_ratio) <= NON_LATIN_MAX_RATIO]
print(f"After cleaning + non-Latin filter: {len(df):,} rows")


## Length filter

This is a short-message autocomplete feature, not long-form generation — drop very short fragments and long argumentative paragraphs (the corpus's political-discussion source has plenty of the latter).


In [ ]:
# ── Length filter ────────────────────────────────────────────
def token_count(text: str) -> int:
    return len(text.split())

df["char_len"]  = df["text"].str.len()
df["token_len"] = df["text"].map(token_count)

df = df[
    (df["char_len"] >= MIN_CHARS) & (df["char_len"] <= MAX_CHARS) &
    (df["token_len"] >= MIN_TOKENS) & (df["token_len"] <= MAX_TOKENS)
]
print(f"After length filter: {len(df):,} rows")


## Deduplicate — exact and near-duplicates

Comment corpora repeat stock phrases constantly ("seri bro", "nalla iruku", etc.). Exact-normalized dedup catches literal repeats; a blocked near-duplicate pass (grouping by the first 3 normalized words, then comparing within each small group) catches near-identical variants without doing an expensive full pairwise comparison across 180k+ rows.


In [ ]:
# ── Deduplication ────────────────────────────────────────────
NORM_RE = re.compile(r"[^a-z0-9\s]")

def normalize(text: str) -> str:
    t = text.lower()
    t = NORM_RE.sub("", t)
    t = WS_RE.sub(" ", t).strip()
    return t

df["norm"] = df["text"].map(normalize)

before = len(df)
df = df.drop_duplicates(subset="norm", keep="first")
print(f"After exact-duplicate removal: {len(df):,} rows (dropped {before - len(df):,})")

# Near-duplicate pass: block by first 3 normalized words, compare within block
def block_key(norm_text: str) -> str:
    return " ".join(norm_text.split()[:3])

df["block"] = df["norm"].map(block_key)

keep_mask = [True] * len(df)
rows = df.reset_index(drop=True)
groups = defaultdict(list)
for idx, key in enumerate(rows["block"]):
    groups[key].append(idx)

dropped_near_dup = 0
for key, idxs in groups.items():
    if len(idxs) < 2:
        continue
    kept_texts = []
    for i in idxs:
        cand = rows.at[i, "norm"]
        is_dup = any(
            difflib.SequenceMatcher(None, cand, kept).ratio() >= NEAR_DUP_THRESHOLD
            for kept in kept_texts
        )
        if is_dup:
            keep_mask[i] = False
            dropped_near_dup += 1
        else:
            kept_texts.append(cand)

rows = rows[keep_mask].reset_index(drop=True)
df = rows
print(f"After near-duplicate removal: {len(df):,} rows (dropped {dropped_near_dup:,} near-dupes)")


## Domain-balance the sources

The corpus's own docs flag heavy skew toward YouTube comments and r/TamilNaduDiscussion political commentary. Cap any single `source` at 10% of the cleaned corpus so autocomplete doesn't over-learn political-argument phrasing at the expense of everyday chat.


In [ ]:
# ── Domain balancing ─────────────────────────────────────────
total_before_balance = len(df)
cap = int(total_before_balance * MAX_SOURCE_SHARE)
print(f"Cap per source: {cap:,} rows (of {total_before_balance:,} total)")

balanced_parts = []
summary = []
for source_val, group in df.groupby("source"):
    n = len(group)
    if n > cap:
        group = group.sample(n=cap, random_state=42)
    balanced_parts.append(group)
    summary.append((source_val, n, len(group)))

df = pd.concat(balanced_parts, ignore_index=True)
df = df.sample(frac=1.0, random_state=42).reset_index(drop=True)  # shuffle

print(f"\nFinal cleaned + balanced dataset: {len(df):,} rows\n")
print("Domain balance summary (source | before cap | after cap):")
for source_val, n_before, n_after in sorted(summary, key=lambda x: -x[1]):
    print(f"  {source_val:<35} {n_before:>8,}  ->  {n_after:>8,}")


In [ ]:
# ── Train/val split ──────────────────────────────────────────
n_val = max(int(len(df) * VAL_FRACTION), 500)
val_df   = df.iloc[:n_val].reset_index(drop=True)
train_df = df.iloc[n_val:].reset_index(drop=True)

print(f"Train: {len(train_df):,} rows")
print(f"Val  : {len(val_df):,} rows")

raw_datasets = DatasetDict({
    "train"     : Dataset.from_pandas(train_df[["text"]], preserve_index=False),
    "validation": Dataset.from_pandas(val_df[["text"]], preserve_index=False),
})
print(raw_datasets)


## Tokenizer


In [ ]:
# ── Tokenizer ────────────────────────────────────────────────
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 family has no pad token — reuse eos_token

print(f"Vocab size : {len(tokenizer)}")
print(f"EOS token  : {tokenizer.eos_token!r}  (id {tokenizer.eos_token_id})")
print(f"PAD token  : {tokenizer.pad_token!r}  (id {tokenizer.pad_token_id})")


## Tokenize — per sentence, with EOS as separator

**This is the fix from v1.** Each sentence is tokenized on its own and gets an explicit EOS token appended, so the model learns "this is where a complete utterance ends" instead of learning that one sentence's end flows straight into an unrelated sentence's start. No fixed-block concatenation. Padding is left to the data collator at batch time (dynamic padding), and `group_by_length` at training time clusters similar-length sequences together to minimize wasted padding compute.


In [ ]:
# ── Per-sentence tokenization with EOS separator ─────────────
def tokenize_fn(batch):
    texts_with_eos = [t + tokenizer.eos_token for t in batch["text"]]
    return tokenizer(
        texts_with_eos,
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,   # dynamic padding at batch time via the data collator
    )

tokenized = raw_datasets.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"],
    desc="Tokenising (per-sentence, EOS-separated)",
)

print(tokenized)
lengths = [len(x) for x in tokenized["train"]["input_ids"][:2000]]
print(f"Sample token-length stats (first 2000 rows): min={min(lengths)} max={max(lengths)} avg={sum(lengths)/len(lengths):.1f}")


## Model


In [ ]:
# ── Load model ───────────────────────────────────────────────
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
model.config.pad_token_id = tokenizer.pad_token_id

n_params = sum(p.numel() for p in model.parameters())
print(f"DistilGPT2 parameters: {n_params:,}")


## Training

Step-based evaluation (every 500 steps) instead of per-epoch, since with ~175k training rows a single epoch is thousands of steps — waiting a full epoch to check eval loss would catch overfitting far too late. Gradient accumulation reaches an effective batch size of 64 without exceeding Kaggle GPU memory at a sequence length of 64.


In [ ]:
# ── Training ─────────────────────────────────────────────────
use_fp16 = DEVICE == "cuda"

training_args = TrainingArguments(
    output_dir                  = OUTPUT_MODEL_DIR,
    overwrite_output_dir        = True,
    num_train_epochs            = NUM_EPOCHS,
    per_device_train_batch_size = PER_DEVICE_TRAIN_BATCH,
    per_device_eval_batch_size  = PER_DEVICE_EVAL_BATCH,
    gradient_accumulation_steps = GRAD_ACCUM_STEPS,
    learning_rate               = LEARNING_RATE,
    warmup_ratio                = WARMUP_RATIO,
    weight_decay                = WEIGHT_DECAY,
    lr_scheduler_type           = "cosine",
    fp16                        = use_fp16,
    group_by_length             = True,   # cluster similar-length sequences to cut padding waste
    # Eval & checkpointing — step-based, not epoch-based
    eval_strategy               = "steps",
    eval_steps                  = EVAL_STEPS,
    save_strategy               = "steps",
    save_steps                  = SAVE_STEPS,
    save_total_limit            = SAVE_TOTAL_LIMIT,
    load_best_model_at_end      = True,
    metric_for_best_model       = "eval_loss",
    greater_is_better           = False,
    # Logging
    logging_dir                 = "/kaggle/working/logs",
    logging_steps               = 100,
    logging_first_step          = True,
    report_to                   = "none",
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer = tokenizer,
    mlm       = False,   # Causal LM — no masking
)

trainer = Trainer(
    model         = model,
    args          = training_args,
    train_dataset = tokenized["train"],
    eval_dataset  = tokenized["validation"],
    tokenizer     = tokenizer,
    data_collator = data_collator,
)

steps_per_epoch = len(tokenized["train"]) // (PER_DEVICE_TRAIN_BATCH * GRAD_ACCUM_STEPS)
print(f"Training on {DEVICE.upper()}  |  fp16={use_fp16}")
print(f"Effective batch size : {PER_DEVICE_TRAIN_BATCH * GRAD_ACCUM_STEPS}")
print(f"Steps per epoch      : {steps_per_epoch:,}")
print(f"Total steps          : {steps_per_epoch * NUM_EPOCHS:,}")

train_result = trainer.train()

print("\n── Loss / perplexity summary ─────────────────────────────")
for log in trainer.state.log_history:
    if "eval_loss" in log:
        step = log.get("step", "?")
        ev_loss = log["eval_loss"]
        ppl = math.exp(ev_loss)
        print(f"  Step {step:>6}  eval_loss={ev_loss:.4f}  perplexity={ppl:.2f}")


In [ ]:
# ── Save model + tokenizer ───────────────────────────────────
os.makedirs(OUTPUT_MODEL_DIR, exist_ok=True)
trainer.save_model(OUTPUT_MODEL_DIR)
tokenizer.save_pretrained(OUTPUT_MODEL_DIR)
print(f"Model saved to: {OUTPUT_MODEL_DIR}")
print("Files:", os.listdir(OUTPUT_MODEL_DIR))


In [ ]:
# ── Zip for easy download ─────────────────────────────────────
print(f"Zipping {OUTPUT_MODEL_DIR} -> {OUTPUT_ZIP} ...")
with zipfile.ZipFile(OUTPUT_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(OUTPUT_MODEL_DIR):
        dirs[:] = [d for d in dirs if not d.startswith("checkpoint-")]
        for file in files:
            full_path = os.path.join(root, file)
            arcname = os.path.relpath(full_path, os.path.dirname(OUTPUT_MODEL_DIR))
            zf.write(full_path, arcname)

zip_size_mb = os.path.getsize(OUTPUT_ZIP) / 1e6
print(f"Done. ZIP size: {zip_size_mb:.1f} MB")
print(f"Download it from: Kaggle Output panel -> {os.path.basename(OUTPUT_ZIP)}")


## Sanity check

Casual, everyday prompts (deliberately avoiding the corpus's heavier political-commentary domain), to eyeball completion quality before downloading.


In [ ]:
# ── Sanity-check: generate completions for sample prompts ────
from transformers import pipeline

generator = pipeline(
    "text-generation",
    model     = OUTPUT_MODEL_DIR,
    tokenizer = OUTPUT_MODEL_DIR,
    device    = 0 if DEVICE == "cuda" else -1,
)

SAMPLE_PROMPTS = [
    "Naan gym ku pogaren",
    "Avan late ah varuvaan",
    "Ippo enna panre bro",
    "Romba naal aachu da",
    "Office la meeting",
    "Nee vandhu",
    "Amma cooking",
    "Weekend trip",
]

print("\n── Sample completions ──────────────────────────────────")
for prompt in SAMPLE_PROMPTS:
    out = generator(
        prompt,
        max_new_tokens     = 15,
        do_sample          = True,
        temperature        = 0.8,
        top_p              = 0.9,
        repetition_penalty = 1.2,
        pad_token_id       = tokenizer.eos_token_id,
    )
    generated = out[0]["generated_text"]
    completion = generated[len(prompt):].strip()
    print(f"  Prompt     : {prompt!r}")
    print(f"  Completion : {completion!r}")
    print()
